# S4.3 - Broadcast join
**Date completed:** November 2026  
**Status:** Completed  
**Key concept:** Send the small table to every executor so the big table is not shuffled

In [0]:
# ============================================================
# S4.3 - Cell 2
# Broadcast Join:
# threshold, automatic broadcast, explicit hint, and plans
#
# Goal:
# 1. Inspect broadcast-related thresholds in this workspace.
# 2. Compare a join plan with and without a broadcast hint.
# 3. Confirm the hint changes strategy, not business results.
# 4. Observe a larger join without assuming the strategy.
# 5. Disable automatic broadcast safely and inspect the plan.
# 6. Show that an explicit broadcast hint can still request
#    broadcast when automatic broadcasting is disabled.
#
# Scope:
# Broadcast joins only.
#
# Deferred:
# - sort-merge / shuffle-hash strategy comparison -> S4.4
# - duplicate/cardinality analysis                -> S4.5
#
# Important:
# explain() shows the optimizer plan.
# With AQE enabled, Spark/Databricks may adapt the strategy
# during execution. Query Profile is useful for runtime proof.
# ============================================================

import pyspark.sql.functions as F

print("=== S4.3: BROADCAST JOIN ===")
print(f"Spark version: {spark.version}")
print()

# ============================================================
# Step 1 - Broadcast Configuration
#
# There are TWO concepts to distinguish:
#
# 1. Normal planning threshold
#
#    spark.sql.autoBroadcastJoinThreshold
#
# 2. Databricks AQE runtime broadcast threshold
#
#    spark.databricks.adaptive.autoBroadcastJoinThreshold
#
# AQE can change a join strategy during execution based on
# runtime statistics.
# ============================================================

print("=== STEP 1: BROADCAST CONFIGURATION ===")


def get_conf_or_none(key):

    try:
    
        return spark.conf.get(key)
    
    except Exception:
    
        return None

static_threshold = get_conf_or_none("spark.sql.autoBroadcastJoinThreshold")

adaptive_threshold = get_conf_or_none("spark.databricks.adaptive.autoBroadcastJoinThreshold")

aqe_enabled = get_conf_or_none("spark.databricks.optimizer.adaptive.enabled")

print("spark.sql.autoBroadcastJoinThreshold              =",static_threshold)

print("spark.databricks.adaptive.autoBroadcastJoinThreshold =",adaptive_threshold)

print("Databricks AQE enabled                            =",aqe_enabled)

print()


In [0]:
# ============================================================
# Step 2 - TPC-H Tables
#
# orders
# → large fact-style table
#
# customer
# → larger dimension/reference table
#
# nation
# → very small lookup table
#
# Row counts are useful context, but broadcast decisions are
# based on estimated SIZE / statistics, not row count alone.
# ============================================================

orders = (
    spark.table("samples.tpch.orders")
    .select(
        "o_orderkey",
        "o_custkey",
        "o_totalprice"))

customer = (
    spark.table("samples.tpch.customer")
    .select(
        "c_custkey",
        "c_nationkey"))

nation = (
    spark.table("samples.tpch.nation")
    .select(
        "n_nationkey",
        "n_name"))

n_orders = orders.count()
n_customer = customer.count()
n_nation = nation.count()

print("=== STEP 2: SOURCE ROW COUNTS ===")

print(f"orders   : {n_orders:,}")

print(f"customer : {n_customer:,}")

print(f"nation   : {n_nation:,}")
print()

In [0]:
# ============================================================
# Step 3 - Small Join WITHOUT Explicit Hint
#
# customer -> nation
#
# nation contains only a very small number of rows.
#
# Prediction:
#
# Spark MAY automatically broadcast nation.
#
# Do not say MUST.
#
# Inspect the plan for something such as:
#
# BroadcastHashJoin
# BroadcastExchange
#
# With AQE, the runtime strategy can also change later.
# ============================================================

nation_condition = (customer["c_nationkey"] == nation["n_nationkey"])

j_auto = (customer.join(nation, nation_condition,"inner"))

print("=== STEP 3: PLAN - NO EXPLICIT HINT ===")

j_auto.explain("formatted")

print()

In [0]:
# ============================================================
# Step 4 - Same Join WITH Explicit Broadcast Hint
#
# F.broadcast(nation)
#
# tells Spark that nation should be considered the broadcast
# side.
#
# For a supported join, look for:
#
# BroadcastHashJoin
#
# and:
#
# BroadcastExchange
#
# in the physical plan.
# ============================================================

j_hint = (customer.join(F.broadcast(nation),nation_condition,"inner"))

print("=== STEP 4: PLAN - F.broadcast(nation) ===")

j_hint.explain("formatted")

print()

In [0]:
# ============================================================
# Step 5 - Result Reconciliation
#
# A join strategy should affect HOW Spark performs the join,
# not the logical business result.
#
# Therefore:
#
# no hint
# and
# explicit broadcast hint
#
# should return the same rows.
# ============================================================

n_auto = (j_auto.count())

n_hint = (j_hint.count())

print("=== STEP 5: RESULT CHECK ===")

print(f"No-hint rows : {n_auto:,}")

print(f"Hint rows    : {n_hint:,}")

print(f"Equal        : {n_auto == n_hint}")

print(f"Customer rows: {n_customer:,}")

print(f"Join == customer rows: "f"{n_auto == n_customer}")

print()


In [0]:
# ============================================================
# Step 6 - Larger Join: Orders -> Customer
#
# IMPORTANT:
#
# Do NOT predict:
#
# "Spark should not broadcast."
#
# The actual strategy depends on:
#
# - estimated relation size
# - available statistics
# - configured thresholds
# - explicit hints
# - AQE/runtime statistics
#
# We observe the plan instead.
# ============================================================

orders_customer_condition = (orders["o_custkey"] == customer["c_custkey"])

j_large = (orders.join(customer,orders_customer_condition,"inner"))

print("=== STEP 6: PLAN - ORDERS x CUSTOMER ===")

j_large.explain("formatted")

print()


In [0]:
# ============================================================
# Step 7 - Disable Automatic Broadcast
#
# We temporarily disable:
#
# 1. Normal Spark automatic broadcast
#    spark.sql.autoBroadcastJoinThreshold
#
# 2. Databricks AQE automatic broadcast
#    spark.databricks.adaptive.autoBroadcastJoinThreshold
#
# Step 9 will restore the original values.
#
# No try block is used here, so there is no incomplete
# try/finally syntax problem.
# ============================================================

print(
    "=== STEP 7: AUTO BROADCAST OFF ==="
)


# ------------------------------------------------------------
# Save original values so Step 9 can restore them.
# ------------------------------------------------------------

original_static_threshold = (
    static_threshold
)

original_adaptive_threshold = (
    adaptive_threshold
)


# ------------------------------------------------------------
# Disable normal Spark automatic broadcast.
# ------------------------------------------------------------

if original_static_threshold is not None:

    spark.conf.set(
        "spark.sql.autoBroadcastJoinThreshold",
        "-1"
    )


# ------------------------------------------------------------
# Disable Databricks AQE automatic broadcast if available.
# ------------------------------------------------------------

if original_adaptive_threshold is not None:

    spark.conf.set(
        "spark.databricks.adaptive.autoBroadcastJoinThreshold",
        "-1"
    )


print(
    "Static threshold while disabled:",
    get_conf_or_none(
        "spark.sql.autoBroadcastJoinThreshold"
    )
)

print(
    "AQE threshold while disabled   :",
    get_conf_or_none(
        "spark.databricks.adaptive.autoBroadcastJoinThreshold"
    )
)


# ------------------------------------------------------------
# Join again WITHOUT an explicit broadcast hint.
#
# Prediction:
#
# Automatic broadcast is disabled.
#
# We inspect the plan instead of assuming the exact
# replacement strategy.
# ------------------------------------------------------------

print()
print("=== STEP 7A: PLAN - AUTO OFF, NO HINT ===")

c4 = (customer.alias("c"))

n4 = (nation.alias("n"))

j_auto_off = (c4.join(n4,F.col("c.c_nationkey") == F.col("n.n_nationkey"),"inner"))

j_auto_off.explain("formatted")

In [0]:
# ============================================================
# Step 8 - Explicit Broadcast Hint While Auto Broadcast Is Off
#
# Automatic broadcast is currently disabled from Step 7.
#
# Here we explicitly request:
#
# F.broadcast(nation)
#
# For this inner equi-join, we expect Spark to use nation
# as the broadcast side even though automatic broadcasting
# is disabled.
#
# Compare:
#
# j_auto_off
# → no explicit hint
#
# j_forced
# → explicit broadcast hint
#
# The execution strategy may differ, but the logical result
# should remain the same.
# ============================================================

print()

print(
    "=== STEP 8: EXPLICIT HINT WITH AUTO BROADCAST OFF ==="
)


j_forced = (
    customer
    .join(
        F.broadcast(nation),
        nation_condition,
        "inner"
    )
)


print(
    "=== PLAN: EXPLICIT BROADCAST HINT ==="
)


j_forced.explain(
    "formatted"
)


# ------------------------------------------------------------
# Execute both joins and compare results.
# ------------------------------------------------------------

n_auto_off = (
    j_auto_off.count()
)


n_forced = (
    j_forced.count()
)


print()

print(
    f"No-hint rows with auto off : "
    f"{n_auto_off:,}"
)

print(
    f"Explicit-hint rows          : "
    f"{n_forced:,}"
)

print(
    f"Results equal               : "
    f"{n_auto_off == n_forced}"
)

In [0]:
# ============================================================
# Step 9 - Restore Original Broadcast Thresholds
#
# On Databricks Serverless, some Spark configuration
# properties may not be exposed to the user.
#
# Therefore:
#
# - restore only configurations that were available
# - read them using get_conf_or_none()
# - do not call spark.conf.get() directly for an
#   unavailable configuration
# ============================================================

print(
    "=== STEP 9: RESTORE THRESHOLDS ==="
)


# ------------------------------------------------------------
# Restore normal Spark threshold only if it was available
# in this environment.
# ------------------------------------------------------------

if original_static_threshold is not None:

    spark.conf.set(
        "spark.sql.autoBroadcastJoinThreshold",
        original_static_threshold
    )


# ------------------------------------------------------------
# Restore Databricks AQE threshold only if it was available.
# ------------------------------------------------------------

if original_adaptive_threshold is not None:

    spark.conf.set(
        "spark.databricks.adaptive.autoBroadcastJoinThreshold",
        original_adaptive_threshold
    )


print()
print(
    "=== STEP 9: THRESHOLDS AFTER RESTORE ==="
)


restored_static_threshold = (
    get_conf_or_none(
        "spark.sql.autoBroadcastJoinThreshold"
    )
)


restored_adaptive_threshold = (
    get_conf_or_none(
        "spark.databricks.adaptive.autoBroadcastJoinThreshold"
    )
)


print(
    "Static threshold =",
    restored_static_threshold
)

print(
    "AQE threshold    =",
    restored_adaptive_threshold
)

# ------------------------------------------------------------
# Explain unavailable values clearly.
# ------------------------------------------------------------

if restored_static_threshold is None:

    print(
        "Static threshold is not exposed "
        "in this Databricks environment."
    )


if restored_adaptive_threshold is None:

    print(
        "AQE threshold is not exposed "
        "in this Databricks environment."
    )

In [0]:
# ============================================================
# Step 10 - Hint Guidance
#
# Explicit broadcast may be UNNECESSARY when Spark already
# identifies the small relation and broadcasts it automatically.
#
# A hint is not an absolute command for every possible join.
# Spark documents hints as strategy suggestions; a particular
# strategy may not support every join type.
#
# Therefore:
#
# hint
# !=
# universal guarantee
# ============================================================

print()
print("=== STEP 10: BROADCAST HINT GUIDANCE ===")

print("""
Case 1
------

No hint
+
Spark already chooses broadcast

→ explicit hint may be unnecessary


Case 2
------

Automatic threshold does not choose broadcast
+
we know the relation is safely small

→ explicit F.broadcast(...) can request broadcast


Case 3
------

Join type / strategy does not support the requested plan

→ Spark may not use the hint


Important:

Do not broadcast a DataFrame merely because it has
fewer rows.

The relevant question is whether the broadcast side
is small enough in memory for every worker.
""")

In [0]:
# ============================================================
# Step 11 - Checkpoint Output
#
# Copy these values / plan observations into the final S4.3
# Markdown summary.
# ============================================================

print("=== STEP 11: S4.3 CHECKPOINT ===")


print(
    f"Static threshold                  : "
    f"{original_static_threshold}")

print(
    f"Databricks AQE threshold          : "
    f"{original_adaptive_threshold}")

print(
    f"Databricks AQE enabled            : "
    f"{aqe_enabled}")

print(
    f"Customer rows                     : "
    f"{n_customer:,}")

print(
    f"Nation rows                       : "
    f"{n_nation:,}")

print(
    f"No-hint customer/nation rows      : "
    f"{n_auto:,}")

print(
    f"Broadcast-hint rows               : "
    f"{n_hint:,}")

print(
    f"Hint/no-hint results equal        : "
    f"{n_auto == n_hint}")

In [0]:
# ============================================================
# Step 12 - Final Learning Summary
# ============================================================

print("=== FINAL LEARNING SUMMARY ===")

print("""
BROADCAST JOIN

Small relation
      ↓
copied to workers
      ↓
large relation stays distributed
      ↓
join locally where possible


MAIN BENEFIT

Avoid shuffling the large relation when a broadcast
join is appropriate.

AUTOMATIC BROADCAST

Spark/Databricks can choose broadcast automatically
when relation size and statistics support it.

EXPLICIT BROADCAST

F.broadcast(df)

→ supplies a broadcast strategy hint

IMPORTANT

Broadcast hints influence execution strategy.

They do NOT change the logical join result.

DATABRICKS AQE

AQE may change join strategy using runtime statistics.

Therefore:

initial explain plan
and
executed runtime plan

can differ.


PLAN CLUES

BroadcastHashJoin
BroadcastExchange

→ evidence that broadcast is being planned/used

CAUTION

Do not broadcast based only on row count.

Consider:

estimated bytes
memory requirements
worker memory
join type
statistics

CORE RULE

Large relation
+
safely small relation

→ broadcast the small side when it avoids an expensive
  large-side shuffle.
""")

# S4.3 — Key Takeaways: Broadcast Join

## 1. What a Broadcast Join Does

```text
Shuffle-based join:
both sides may need redistribution by the join key

Broadcast join:
the small relation is copied to workers
so the large relation can avoid a large join-key shuffle
```

The strategy changes **HOW Spark executes the join**, not **WHAT rows the logical join returns**.

---

## 2. When to Use It

A common broadcast scenario is:

```text
Large fact table
+
Safely small lookup / dimension table
```

Broadcast the small side when it can safely fit in worker memory.

Important:

```text
Do not decide from row count alone.
```

Consider:

- estimated size in bytes
- row width
- worker memory
- available statistics
- join type

Broadcasting a relation that is too large can create memory pressure or failure.

---

## 3. Automatic vs Explicit Broadcast

Spark can choose a broadcast join automatically when its optimizer determines that a relation is small enough.

An explicit hint can be supplied with:

```python
F.broadcast(df)
```

This requests the broadcast strategy.

If Spark already chooses the same broadcast strategy automatically, the explicit hint may not change the plan.

A hint is a strategy request and is not a universal guarantee for every possible join.

---

## 4. Reading the Plan in Our Photon Serverless Workspace

In our observed Photon plans, broadcast appeared as:

```text
PhotonBroadcastHashJoin
```

The broadcast side was associated with:

```text
EXECUTOR_BROADCAST
```

A shuffle-based join appeared as:

```text
PhotonShuffledHashJoin
```

with:

```text
hashpartitioning(...)
```

on both sides.

Our inspected plans showed:

```text
isFinalPlan=false
```

Therefore those plans were not yet the final AQE runtime plans.

AQE may adapt the physical plan during execution.

---

## 5. Observed in Our Workspace

Environment:

```text
Spark      : 4.2.0
Compute    : Databricks Serverless
Execution  : Photon
```

Source row counts:

```text
orders      7,500,000
customer      750,000
nation              25
```

### customer × nation — no explicit hint

Observed:

```text
PhotonBroadcastHashJoin
nation was the broadcast side
```

Result:

```text
750,000 rows
```

---

### customer × nation — F.broadcast(nation)

Observed:

```text
PhotonBroadcastHashJoin
```

The plan was effectively the same as the no-hint version.

Result:

```text
750,000 rows
```

Therefore:

```text
no-hint rows = hint rows = customer rows
```

For this join, Spark already selected broadcast automatically, so the explicit hint was unnecessary.

---

### orders × customer

Observed:

```text
PhotonShuffledHashJoin
```

Both sides showed:

```text
hashpartitioning(..., 348)
```

This demonstrates the type of data redistribution that a suitable broadcast join can avoid.

Detailed physical join strategies are covered in **S4.4**.

---

## 6. What Was Not Verified

### Auto-Broadcast Threshold

The workspace did not expose:

```text
spark.sql.autoBroadcastJoinThreshold
```

through our configuration check.

Therefore the actual default auto-broadcast threshold for this Serverless session was **not verified by our notebook**.

We should not record a remembered or documented default as an observed workspace value.

---

### Automatic Broadcast Disabled

We attempted to set the automatic broadcast threshold to:

```text
-1
```

but the subsequent plan still used a broadcast join.

Because the configuration was not exposed in this Serverless environment, we could not verify that the setting actually took effect.

Therefore we did **not** prove:

```text
automatic broadcast successfully disabled
```

and we also did **not** prove:

```text
explicit broadcast hint works while automatic broadcast is disabled
```

That experiment remains inconclusive for this workspace.

---

## 7. Checkpoint

### Q1. What is a broadcast join?

A small relation is copied to workers so the large relation can avoid an expensive join-key shuffle.

---

### Q2. Which side should normally be broadcast?

The safely small side.

Broadcasting a large relation can create memory pressure.

---

### Q3. Does a broadcast hint change the logical result?

No.

It influences the physical execution strategy.

Observed:

```text
no hint = 750,000 rows
hint    = 750,000 rows
```

---

### Q4. When can an explicit broadcast hint be unnecessary?

When Spark already chooses the same broadcast strategy automatically.

That happened in our:

```text
customer × nation
```

join.

---

### Q5. What indicated broadcast in our Photon plan?

```text
PhotonBroadcastHashJoin
EXECUTOR_BROADCAST
```

---

### Q6. What indicated a shuffle-based join in our larger example?

```text
PhotonShuffledHashJoin
hashpartitioning(..., 348)
```

on both sides.

---

### Q7. Did we verify the Serverless auto-broadcast threshold?

No.

The configuration was not exposed by our workspace.

---

## 8. Core Understanding

```text
Large relation
+
Safely small relation
        ↓
Broadcast small side
        ↓
Potentially avoid large join-key shuffle
        ↓
Faster join when appropriate
```

But:

```text
Spark may already choose broadcast automatically.
```

Therefore always inspect the plan before adding hints unnecessarily.

**S4.3 — Broadcast Join COMPLETED.**